In [249]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [250]:
# portfolio = pd.read_csv('../1data/portfolio.csv')
# profile = pd.read_csv('../1data/profile.csv')
# transcript = pd.read_csv('../1data/transcript.csv')

# merged = pd.read_csv('../1data/all_merged.csv')
# promotion = pd.read_csv('../1data/promotion_df.csv')
# normal = pd.read_csv('../1data/normal_flow_df.csv')

portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')

# merged = pd.read_csv('../data/all_merged.csv')
# promotion = pd.read_csv('../data/promotion_df.csv')
# normal = pd.read_csv('../data/normal_flow_df.csv')

In [251]:
def check(df, name="Data"):
    print(f"{name} 기본 정보")
    
    print("\n[1] 데이터 크기")
    display(df.shape)
    
    print("\n[2] 컬럼 정보")
    df.info()
    
    print("\n[3] 결측치 개수")
    display(
    df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("결측치 개수")
    )
    
    print("\n[4] 중복 데이터 개수")
    display(df.duplicated().sum())

---

# profile

In [252]:
# 데이터 타입 date형식으로 변환
profile["became_member_on"] = pd.to_datetime(profile["became_member_on"], format="%Y%m%d")

In [253]:
# 필요없는 Unnamed:0 컬럼 제거
profile = profile.drop('Unnamed: 0', axis=1)

---

# portfolio

In [254]:
# channels마다 파생변수 생성
portfolio['web'] = portfolio['channels'].astype(str).str.contains('web').astype(int)
portfolio['email'] = portfolio['channels'].astype(str).str.contains('email').astype(int)
portfolio['mobile'] = portfolio['channels'].astype(str).str.contains('mobile').astype(int)
portfolio['social'] = portfolio['channels'].astype(str).str.contains('social').astype(int)

# 기존 channels 컬럼 제거
portfolio = portfolio.drop('channels', axis=1)

In [255]:
# portfolio 테이블의 필요없는 인덱스 컬럼 제거
portfolio = portfolio.drop('Unnamed: 0', axis=1)

---

# transcript

In [256]:
# 딕셔너리처럼 생긴 문자열을 진짜 딕셔너리로 변환
transcript['value'] = transcript['value'].apply(ast.literal_eval)

# 딕셔너리의 키 -> 새로운 컬럼
value_df = pd.DataFrame(transcript['value'].tolist())
transcript = pd.concat([transcript, value_df], axis=1)

# offer id를 offer_id로 컬럼명 통일
transcript['offer_id'] = transcript['offer_id'].fillna(transcript['offer id'])

# offer id 컬럼 제거
transcript = transcript.drop('offer id', axis=1)

# value 컬럼 제거
transcript = transcript.drop('value', axis=1)

---

# transcript x profile

In [257]:
# transcript 기준으로 profile 데이터를 left Join
merged_df = pd.merge(
    transcript,
    profile,
    left_on='person',
    right_on='id',
    how='left'
)

# 필요 없는 id 컬럼(person과 중복)은 버리기
merged_df = merged_df.drop(columns='id')

In [258]:
# 결측치 처리
# gender의 결측치 'Unknown'으로 채우기 
merged_df['gender'] = merged_df['gender'].fillna('Unknown')

# age의 118을 결측치(NaN)로 바꿔주기 
merged_df['age'] = merged_df['age'].replace(118, np.nan)

# income은 이미 결측치(NaN) 상태

---

# all_merged

In [259]:
all_merged_df = pd.merge(
    merged_df,
    portfolio,
    left_on='offer_id',
    right_on='id',
    how='left'
)

all_merged_df = all_merged_df.drop(columns='id')

# reward 컬럼명 변경(명확하게)
all_merged_df = all_merged_df.rename(columns={
    "reward_x": "transcript_reward",
    "reward_y": "portfolio_reward"
})

In [260]:
# offer_id 이름 변경 (쿠폰명_difficulty_reward_duration)
portfolio['offer_name'] = (
    portfolio['offer_type'] + '_' + 
    portfolio['difficulty'].astype(str) + '_' + 
    portfolio['reward'].astype(str) + '_' + 
    portfolio['duration'].astype(str)
)
# id : key, offer_name : value
offer_name_dict = portfolio.set_index('id')['offer_name'].to_dict()
all_merged_df['offer_id'] = all_merged_df['offer_id'].map(offer_name_dict)


# 사람(person)별로 먼저 묶고, 그 안에서 시간(time) 순서대로 오름차순 정렬
all_merged_df = all_merged_df.sort_values(by=['person', 'time', 'Unnamed: 0']) # - Unnamed: 0 순서 추가

In [261]:
check(all_merged_df, 'all_merged_df.csv')

all_merged_df.csv 기본 정보

[1] 데이터 크기


(306534, 19)


[2] 컬럼 정보
<class 'pandas.DataFrame'>
Index: 306534 entries, 55972 to 289924
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Unnamed: 0         306534 non-null  int64         
 1   person             306534 non-null  str           
 2   event              306534 non-null  str           
 3   time               306534 non-null  int64         
 4   amount             138953 non-null  float64       
 5   offer_id           167581 non-null  str           
 6   transcript_reward  33579 non-null   float64       
 7   gender             306534 non-null  str           
 8   age                272762 non-null  float64       
 9   became_member_on   306534 non-null  datetime64[us]
 10  income             272762 non-null  float64       
 11  portfolio_reward   167581 non-null  float64       
 12  difficulty         167581 non-null  float64       
 13  duration           167581 non-null  float64  

,결측치 개수
transcript_reward,272955
amount,167581
offer_id,138953
mobile,138953
email,138953
social,138953
web,138953
offer_type,138953
duration,138953
difficulty,138953



[4] 중복 데이터 개수


np.int64(0)

---

# promotion

In [262]:
# 조건 1: 쿠폰 타입이 bogo, discount, informational 인 것
cond_offers = all_merged_df['offer_type'].isin(['bogo', 'discount', 'informational'])

# 조건 2: 이벤트 종류가 transaction(결제) 인 것
cond_transactions = all_merged_df['event'] == 'transaction'

target_df = all_merged_df[cond_offers | cond_transactions].copy()

print(target_df['offer_type'].value_counts(dropna=False))
print(target_df['event'].value_counts(dropna=False))
display(target_df.head())
display(target_df.shape)

offer_type
NaN              138953
bogo              71617
discount          69898
informational     26066
Name: count, dtype: int64
event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64


,Unnamed: 0,person,event,time,amount,offer_id,transcript_reward,gender,age,became_member_on,income,portfolio_reward,difficulty,duration,offer_type,web,email,mobile,social
55972,55972,0009655768c64bdeb2e877511632db8f,offer received,168,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
77705,77705,0009655768c64bdeb2e877511632db8f,offer viewed,192,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
89291,89291,0009655768c64bdeb2e877511632db8f,transaction,228,22.16,NaN,NaN,M,33.0,2017-04-21,72000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113605,113605,0009655768c64bdeb2e877511632db8f,offer received,336,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0
139992,139992,0009655768c64bdeb2e877511632db8f,offer viewed,372,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0


(306534, 19)

In [263]:
# person당 offer_id를 하나의 행으로 설정하여, 흩어진 고객 행동의 순서를 보기 편하게 해주는 "피벗테이블 생성 코드"

# 1. 피벗을 돌릴 '쿠폰 이력서' 데이터만 빼내기
offers_df = target_df[target_df['event'] != 'transaction'].copy()

# 2. 안전한 금고에 보관할 '순수 영수증' 데이터만 빼내기
transactions_df = target_df[target_df['event'] == 'transaction'].copy()

print(f"피벗할 쿠폰 데이터: {len(offers_df)} 줄")
print(f"금고에 보관한 영수증: {len(transactions_df)} 줄")

피벗할 쿠폰 데이터: 167581 줄
금고에 보관한 영수증: 138953 줄


In [ ]:
# 1. 시간 순서대로 예쁘게 줄 세우기
offers_df = offers_df.sort_values(['person', 'offer_id', 'time'])

# 2. 'received' 이벤트가 등장할 때마다 1, 아니면 0인 깃발(Flag) 만들기
offers_df['is_received'] = (offers_df['event'] == 'offer received').astype(int)

# 3. 사람과 쿠폰 단위로 묶어서, 깃발을 누적해서 더하기 (Cumsum)
offers_df['offer_cycle'] = offers_df.groupby(['person', 'offer_id'])['is_received'].cumsum()

# 4. 피벗 돌리기
pivot_df = offers_df.pivot_table(
    index=['person', 'offer_id', 'offer_cycle'],
    columns='event',
    values='time',
    aggfunc='min'
).reset_index()

pivot_df.columns.name = None
pivot_df = pivot_df[['person', 'offer_id', 'offer_cycle', 'offer received', 'offer viewed', 'offer completed']]

# reward 따로 뽑아서 merge <- 여기서 중복이 생기면서 초기 final_df와 차이가 생겼던 것
# reward_df = offers_df[offers_df['event'] == 'offer completed'][['person', 'offer_id', 'offer_cycle', 'transcript_reward']]
# pivot_df = pivot_df.merge(reward_df, on=['person', 'offer_id', 'offer_cycle'], how='left')

# portfolio 정보 붙이기
pivot_df = pivot_df.merge(
    portfolio[['offer_name', 'offer_type', 'difficulty', 'reward', 'duration', 'web', 'email', 'mobile', 'social']],
    left_on='offer_id',
    right_on='offer_name',
    how='left'
).drop(columns='offer_name')

# profile 정보 붙이기
pivot_df = pivot_df.merge(
    profile[['id', 'gender', 'age', 'became_member_on', 'income']],
    left_on='person',
    right_on='id',
    how='left'
).drop(columns='id')

display(pivot_df.head())
display(pivot_df.shape)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,transcript_reward,offer_type,difficulty,reward,duration,web,email,mobile,social,gender,age,became_member_on,income
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,1,408.0,456.0,414.0,5.0,bogo,5,5,5,1,1,1,1,M,33,2017-04-21,72000.0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,1,504.0,540.0,528.0,2.0,discount,10,2,10,1,1,1,1,M,33,2017-04-21,72000.0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,1,576.0,NaN,576.0,2.0,discount,10,2,7,1,1,1,0,M,33,2017-04-21,72000.0
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,1,168.0,192.0,NaN,NaN,informational,0,0,3,0,1,1,1,M,33,2017-04-21,72000.0
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,1,336.0,372.0,NaN,NaN,informational,0,0,4,1,1,1,0,M,33,2017-04-21,72000.0


(76755, 19)

In [265]:
# 1. 원본에서 offer_id와 offer_type 짝꿍 사전 만들기
offer_dict = offers_df[['offer_id', 'offer_type']].drop_duplicates().set_index('offer_id')['offer_type'].to_dict()

# 2. 피벗 테이블의 offer_id를 보고, 임시로 쿠폰 타입(bogo, discount)을 가져오기
temp_offer_type = pivot_df['offer_id'].map(offer_dict)

# 3. 기존 숫자였던 'offer_cycle' 컬럼 위에 곧바로 덮어쓰기
pivot_df['offer_cycle'] = temp_offer_type.str.capitalize() + '_' + pivot_df['offer_cycle'].astype(str)

In [ ]:
# 피벗테이블에 amount 붙이기

# 1. 금고(transactions_df)에서 영수증 알맹이만 꺼내기
transactions_df = transactions_df[['person', 'time', 'amount']]

# 2. 피벗 테이블(pivot_df)에 영수증(receipts) 1:1 도킹하기
final_df = pivot_df.merge(
    transactions_df,
    left_on=['person', 'offer completed'],
    right_on=['person', 'time'],
    how='left'
).drop(columns='time')

# 3. 태블로 퍼널 시각화를 위한 0/1 파생변수 생성
final_df['is_received'] = final_df['offer received'].notna().astype(int)
final_df['is_viewed'] = final_df['offer viewed'].notna().astype(int)
final_df['is_completed'] = final_df['offer completed'].notna().astype(int)

# 4. 정상 흐름 플래그
final_df['is_normal_flow'] = (
    final_df['offer received'].notna() &
    final_df['offer viewed'].notna() &
    final_df['offer completed'].notna() &
    (final_df['offer received'] <= final_df['offer viewed']) &
    (final_df['offer viewed'] <= final_df['offer completed'])
).astype(int)

# 5. 더블카운팅 처리 플래그
final_df['tx_key'] = final_df['person'].astype(str) + '_' + final_df['offer completed'].astype(str)
final_df['is_valid_view'] = final_df['offer viewed'] <= final_df['offer completed']

final_df = final_df.sort_values(
    by=['tx_key', 'is_valid_view', 'offer viewed'],
    ascending=[True, False, False]
)
# final_df = final_df.sort_values(
#     by=['tx_key', 'offer viewed'],
#     ascending=[True, False]
# )

final_df['primary_flag'] = ~final_df.duplicated(subset=['tx_key'], keep='first')
final_df['is_deduplicated'] = (final_df['primary_flag'] & final_df['is_valid_view']).astype(int)
# final_df['is_deduplicated'] = final_df['primary_flag'].astype(int)

promo_total_revenue = final_df.loc[final_df['is_deduplicated'] == 1, 'amount'].sum()

final_df = final_df.drop(columns=['tx_key', 'is_valid_view', 'primary_flag'])
# final_df = final_df.drop(columns=['tx_key', 'primary_flag'])

# print(f"[최종] 프로모션 귀속 총매출액: ${promo_total_revenue:,.2f}")
# print(f"[최종] 귀속된 거래 수: {final_df['is_deduplicated'].sum():,}")
display(final_df.head())
display(final_df.shape)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,transcript_reward,offer_type,difficulty,reward,...,gender,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,5.0,bogo,5,5,...,M,33,2017-04-21,72000.0,8.57,1,1,1,0,0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,2.0,discount,10,2,...,M,33,2017-04-21,72000.0,14.11,1,1,1,0,0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,2.0,discount,10,2,...,M,33,2017-04-21,72000.0,10.27,1,0,1,0,0
9,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,3.0,discount,7,3,...,O,40,2018-01-09,57000.0,11.93,1,1,1,1,1
7,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,5.0,bogo,5,5,...,O,40,2018-01-09,57000.0,22.05,1,1,1,1,1


(76755, 25)

정상 흐름 22333
더블 카운팅 23600 -> 25000


In [279]:
print(f"is_deduplicated == 1 (유효한 행): {(final_df['is_deduplicated'] == 1).sum():,}")
print(f"is_deduplicated == 0 (제거된 행): {(final_df['is_deduplicated'] == 0).sum():,}")
print(f"전체: {len(final_df):,}")

is_deduplicated == 1 (유효한 행): 22,333
is_deduplicated == 0 (제거된 행): 54,422
전체: 76,755


In [283]:
print(f"offer completed 없는 행: {final_df['offer completed'].isna().sum():,}")
print(f"offer completed 있는 행: {final_df['offer completed'].notna().sum():,}")
print(f"offer completed 있는데 is_deduplicated == 0인 행: {((final_df['is_deduplicated'] == 0) & final_df['offer completed'].notna()).sum():,}")

offer completed 없는 행: 43,176
offer completed 있는 행: 33,579
offer completed 있는데 is_deduplicated == 0인 행: 11,246


In [267]:
check(final_df, 'promotion_df.csv')

promotion_df.csv 기본 정보

[1] 데이터 크기


(76755, 25)


[2] 컬럼 정보
<class 'pandas.DataFrame'>
Index: 76755 entries, 0 to 76744
Data columns (total 25 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   person             76755 non-null  str           
 1   offer_id           76755 non-null  str           
 2   offer_cycle        76755 non-null  str           
 3   offer received     76755 non-null  float64       
 4   offer viewed       58090 non-null  float64       
 5   offer completed    33579 non-null  float64       
 6   transcript_reward  33579 non-null  float64       
 7   offer_type         76755 non-null  str           
 8   difficulty         76755 non-null  int64         
 9   reward             76755 non-null  int64         
 10  duration           76755 non-null  int64         
 11  web                76755 non-null  int64         
 12  email              76755 non-null  int64         
 13  mobile             76755 non-null  int64         
 14  social     

,결측치 개수
transcript_reward,43176
offer completed,43176
amount,43176
offer viewed,18665
gender,9810
income,9810
offer received,0
person,0
offer_id,0
difficulty,0



[4] 중복 데이터 개수


np.int64(231)

---

# 확인

더블 카운팅이 정상 흐름에서만 처리가 된 것 같아서 다른 흐름에서의 더블 카운팅도 체크해봐야할듯

In [268]:
promotion = pd.read_csv('../data/promotion_df.csv')

In [269]:
print(f"전체 행 수: {len(promotion):,}")
print(f"정상 흐름(is_normal_flow) 행 수: {promotion['is_normal_flow'].sum():,}")
print(f"더블카운팅 제거 후(is_deduplicated) 행 수: {promotion['is_deduplicated'].sum():,}")

both = (promotion['is_normal_flow'] == 1) & (promotion['is_deduplicated'] == 1)
print(f"정상 흐름 + 더블카운팅 제거: {both.sum():,}")

print(f"차이: {(promotion['is_normal_flow'].sum() - promotion['is_deduplicated'].sum()):,}")

전체 행 수: 76,755
정상 흐름(is_normal_flow) 행 수: 23,519
더블카운팅 제거 후(is_deduplicated) 행 수: 22,333
정상 흐름 + 더블카운팅 제거: 22,333
차이: 1,186


In [270]:
print(f"전체 행 수: {len(final_df):,}")
print(f"정상 흐름(is_normal_flow) 행 수: {final_df['is_normal_flow'].sum():,}")
print(f"더블카운팅 제거 후(is_deduplicated) 행 수: {final_df['is_deduplicated'].sum():,}")

both = (final_df['is_normal_flow'] == 1) & (final_df['is_deduplicated'] == 1)
print(f"정상 흐름 + 더블카운팅 제거: {both.sum():,}")

print(f"차이: {(final_df['is_normal_flow'].sum() - final_df['is_deduplicated'].sum()):,}")

전체 행 수: 76,755
정상 흐름(is_normal_flow) 행 수: 23,519
더블카운팅 제거 후(is_deduplicated) 행 수: 22,333
정상 흐름 + 더블카운팅 제거: 22,333
차이: 1,186


정상 흐름만 먼저 뽑아내고 거기서 더블 카운팅을 제거해본다면?

In [271]:
# 1) 두 조건의 개수 확인
count_true_conversion = promotion['is_normal_flow'].sum()
count_valid_attr = promotion['is_deduplicated'].sum()
diff = count_true_conversion - count_valid_attr

print(f"is_true_conversion 합계: {count_true_conversion:,}")
print(f"valid_attribution_flag 합계: {count_valid_attr:,}")
print(f"차이: {diff:,}")

# 2) is_true_conversion=1 이지만 valid_attribution_flag=0 인 행만 추출
# - last touch로 선택되지 못한 값들 선택
lost_rows = promotion[
    (promotion['is_normal_flow'] == 1) &
    (promotion['is_deduplicated'] == 0)
].copy()

print(f"전환으로는 잡혔지만 최종 귀속에서는 빠진 행 수: {len(lost_rows):,}")

# 3) 검증
assert len(lost_rows) == diff, "차이 수와 실제 빠진 행 수가 다름"

display(lost_rows.head())

is_true_conversion 합계: 23,519
valid_attribution_flag 합계: 22,333
차이: 1,186
전환으로는 잡혔지만 최종 귀속에서는 빠진 행 수: 1,186


,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,transcript_reward,offer_type,difficulty,reward,...,gender,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated
5,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,5.0,discount,20,5,...,O,40,2018-01-09,57000.0,22.05,1,1,1,1,0
48,00ae03011f9f49b8a4b3e6d416678b0b,bogo_10_10_7,Bogo_2,504.0,534.0,618.0,10.0,bogo,10,10,...,M,55,2015-11-15,83000.0,30.83,1,1,1,1,0
81,00cf1bbec83f4a658f8994e556db4633,discount_10_2_10,Discount_2,336.0,438.0,564.0,2.0,discount,10,2,...,M,53,2018-05-15,73000.0,33.28,1,1,1,1,0
153,015c3d28c67e46aa95e9ec97c27220e8,discount_10_2_7,Discount_1,504.0,510.0,618.0,2.0,discount,10,2,...,M,56,2016-03-22,99000.0,22.12,1,1,1,1,0
160,01633b71b3a2457aa7d35d8bcc3afb5a,discount_20_5_10,Discount_1,504.0,552.0,600.0,5.0,discount,20,5,...,M,62,2015-09-02,90000.0,24.76,1,1,1,1,0


---

# normal_flow

In [272]:
# # 정상 흐름 필터링: received <= viewed <= completed (모두 존재해야 함)
# normal_flow_df = final_df[
#     final_df['offer received'].notna() &
#     final_df['offer viewed'].notna() &
#     final_df['offer completed'].notna() &
#     (final_df['offer received'] <= final_df['offer viewed']) &
#     (final_df['offer viewed'] <= final_df['offer completed'])
# ].copy()

# display(normal_flow_df.head())
# display(normal_flow_df.shape)

In [273]:
# normal_flow_df.to_csv('../data/normal_flow_df.csv', index=False)

In [274]:
# check(normal, 'normal_flow_df.csv')

---

In [275]:
# all_merged_df.to_csv('../data/all_merged.csv', index=False)
# final_df.to_csv('../data/promotion_df.csv', index=False)